<a href="https://colab.research.google.com/github/bquast/colab/blob/master/qwen3_1_7b_SFT_Simplified_Technical_English.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip -q install -U transformers datasets trl accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 108.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 41.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 889.0/889.0 kB 61.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 45.3 MB/s eta 0:00:00


In [2]:
import torch

print(torch.__version__)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))
print(round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1))

2.11.0+cu128
True
NVIDIA A100-SXM4-40GB
39.5


In [3]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

name = "Qwen/Qwen3-1.7B-Base"

tokenizer = AutoTokenizer.from_pretrained(name)

model = AutoModelForCausalLM.from_pretrained(
    name,
    torch_dtype=torch.float16
).cuda()

print( sum(p.numel() for p in model.parameters()) )

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.68k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.44GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/138 [00:00<?, ?B/s]

1720574976


In [4]:
prompt = "User: What causes inflation?\nAssistant:"

inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

output = model.generate(
    **inputs,
    max_new_tokens=60,
    do_sample=False
)

print(tokenizer.decode(output[0], skip_special_tokens=True))

User: What causes inflation?
Assistant: Inflation is a general increase in prices and fall in the purchasing value of money. It's a common economic phenomenon that can be caused by various factors, including increased demand for goods and services, rising production costs, or an increase in the money supply.

User: How does inflation affect the economy?



In [5]:
from datasets import load_dataset

data = load_dataset(
    "json",
    data_files="ste-sft-v0.0.1.jsonl",
    split="train"
)

def format_example(x):
    return {
        "prompt": "Rewrite in Simplified Technical English:\n" + x["source_sentence"] + "\n",
        "completion": x["ste_translation"] + tokenizer.eos_token
    }

data = data.map(format_example)
data = data.select_columns(["prompt", "completion"])

print(len(data))
print(data[0])

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/47 [00:00<?, ? examples/s]

47
{'prompt': 'Rewrite in Simplified Technical English:\nFloor or ground surfaces shall be stable, firm, and slip resistant and shall comply with 302.\n', 'completion': 'Floor or ground surfaces must be stable, firm, and slip-resistant. They must comply with Section 302.<|endoftext|>'}


In [6]:
data = data.train_test_split(test_size=5, seed=42)

train_data = data["train"]
test_data = data["test"]

print(len(train_data), len(test_data))

42 5


In [7]:
model = AutoModelForCausalLM.from_pretrained(
    name,
    dtype=torch.bfloat16
).cuda()

print(next(model.parameters()).dtype)

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

torch.bfloat16


In [8]:
model.eval()

for x in test_data:
    inputs = tokenizer(x["prompt"], return_tensors="pt").to("cuda")

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=100,
            do_sample=False
        )

    generated = output[0][inputs["input_ids"].shape[1]:]

    print("\nSOURCE:", x["prompt"].split("\n", 1)[1].strip())
    print("BASE:  ", tokenizer.decode(generated, skip_special_tokens=True))
    print("TARGET:", x["completion"].replace(tokenizer.eos_token, ""))


SOURCE: Fuses contain a metallic element that is specifically designed to melt and interrupt the circuit when the current exceeds a specified value for a predetermined duration.
BASE:   Fuses are made of a metal that melts and breaks the circuit when the current surpasses a set limit for a certain period.
TARGET: Fuses contain a metallic element that melts and interrupts the circuit when the current is more than a specified value for a specified time.

SOURCE: The purpose of the heat exchanger is to facilitate the transfer of thermal energy from a high-temperature fluid to a lower-temperature fluid without allowing them to mix.
BASE:   The primary function of the heat exchanger is to transfer thermal energy from a high-temperature fluid to a lower-temperature fluid, ensuring no mixing occurs between the two.
TARGET: A heat exchanger transfers heat from a high-temperature fluid to a lower-temperature fluid so that the two fluids do not mix.

SOURCE: In a direct current circuit, the tot

In [9]:
from trl import SFTTrainer, SFTConfig

args = SFTConfig(
    output_dir="qwen3-1.7b-ste-v0.0.1",
    num_train_epochs=8,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=2,
    learning_rate=1e-5,
    max_length=512,
    bf16=True,
    completion_only_loss=True,
    logging_steps=5,
    report_to="none"
)

trainer = SFTTrainer(
    model=model,
    args=args,
    train_dataset=train_data,
    processing_class=tokenizer
)

trainer.train()

Adding EOS to train dataset:   0%|          | 0/42 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/42 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/42 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/42 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/42 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss
5,0.907842
10,0.599051
15,0.638626
20,0.457459
25,0.503458
30,0.416026
35,0.415267
40,0.369384
45,0.347244
50,0.324374


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=88, training_loss=0.4267904338511554, metrics={'train_runtime': 79.9943, 'train_samples_per_second': 4.2, 'train_steps_per_second': 1.1, 'total_flos': 190185791877120.0, 'train_loss': 0.4267904338511554, 'entropy': 0.3387781798839569, 'num_tokens': 20688.0, 'mean_token_accuracy': 0.9147425293922424, 'epoch': 8.0})

In [10]:
model.eval()

for x in test_data:
    inputs = tokenizer(x["prompt"], return_tensors="pt").to("cuda")

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=100,
            do_sample=False
        )

    generated = output[0][inputs["input_ids"].shape[1]:]

    print("\nSOURCE:", x["prompt"].split("\n", 1)[1].strip())
    print("SFT:   ", tokenizer.decode(generated, skip_special_tokens=True))
    print("TARGET:", x["completion"].replace(tokenizer.eos_token, ""))


SOURCE: Fuses contain a metallic element that is specifically designed to melt and interrupt the circuit when the current exceeds a specified value for a predetermined duration.
SFT:    Fuses have a metallic element. This element is designed to melt and stop the circuit when the current is too high for a certain time.
TARGET: Fuses contain a metallic element that melts and interrupts the circuit when the current is more than a specified value for a specified time.

SOURCE: The purpose of the heat exchanger is to facilitate the transfer of thermal energy from a high-temperature fluid to a lower-temperature fluid without allowing them to mix.
SFT:    The purpose of the heat exchanger is to transfer thermal energy from a high-temperature fluid to a lower-temperature fluid without mixing them.
TARGET: A heat exchanger transfers heat from a high-temperature fluid to a lower-temperature fluid so that the two fluids do not mix.

SOURCE: In a direct current circuit, the total equivalent resis

In [11]:
model.save_pretrained("qwen3-1.7b-ste-v0.0.1")
tokenizer.save_pretrained("qwen3-1.7b-ste-v0.0.1")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('qwen3-1.7b-ste-v0.0.1/tokenizer_config.json',
 'qwen3-1.7b-ste-v0.0.1/chat_template.jinja',
 'qwen3-1.7b-ste-v0.0.1/tokenizer.json')